In [370]:
using LowLevelFEM

In [371]:
openGeometry("beam-3D.geo")
#openPreProcessor()

The hinge coordinates are stored as Gmsh parameters and can be recovered directly from the model. They can be useful later for probing or plotting results at the hinge locations.


## Fields and materials

Each beam segment is associated with a `Material` object. Two unknown fields are introduced:

- `U`: the two-component translational displacement field,
- `Φ`: the scalar rotation field.

The rotation field is created with `reducedOrder=true`, so it is solved in a one-order-lower continuous Lagrange space while assembly and storage remain associated with the original mesh.


In [372]:
mat = Material("beam")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u, rhsName=:f)
Φ = Field([mat], type=:VectorField, dim=3, fieldName=:φ, rhsName=:m, reducedOrder=false);

In [373]:
t = tangentVector(U, "beam")
nξ, nη = normalVectors(U, "beam");


In [374]:
t = [t[1], t[2], t[3]]
nξ = [nξ[1], nξ[2], nξ[3]]
nη = [nη[1], nη[2], nη[3]]

3-element Vector{ScalarField}:
 ScalarField([[0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;]  …  [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;], [1.0; 1.0;;]], Matrix{Float64}(undef, 0, 0), [0.0], [5, 6, 7, 8, 9, 10, 11, 12, 13, 14  …  25, 26, 27, 28, 29, 30, 31, 32, 33, 34], 1, :scalar, Problem("beam-3D", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 31, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :u, :f, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), :field)
 ScalarField([[0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;], [0.0; 0.0;;]  …  [0.0; 0.0;;], 

## Timoshenko beam kinematics

The generalized beam strain vector is

$$
\boldsymbol{\varepsilon}_b=
\begin{bmatrix}
\varepsilon\\
\gamma\\
\kappa_b
\end{bmatrix},
$$

with axial strain

$$
\varepsilon=\mathbf{t}^T(\nabla\mathbf{u})\mathbf{t},
$$

shear strain

$$
\gamma=\mathbf{n}^T(\nabla\mathbf{u})\mathbf{t}-\varphi,
$$

and curvature

$$
\kappa_b=\nabla\varphi\cdot\mathbf{t}.
$$

The matrices `Au`, `Aφ`, and `Gφ` map the LowLevelFEM operator output to these generalized strain components.


In [375]:
# ------------------------------------------------------------------
# Generalized Timoshenko beam strain
#
# ε = tᵀ ∇u t
# γ = nᵀ ∇u t - φ
# κ = ∇φ ⋅ t
# ------------------------------------------------------------------

Au = vcat(kron(nξ,t)',
          kron(nη,t)',
          kron(t,t)',
          zeros(3,9)
          )

Aφ = vcat(zeros(3,9),
          kron(nξ,t)',
          kron(nη,t)',
          kron(t,t)'
          )

Gφ = vcat(-nη',
          nξ',
          zeros(4,3)
          )

Bu = Au ⋅ Grad(U)

Bφ = Aφ ⋅ Grad(Φ) + Gφ ⋅ Φ;

In [376]:
d = 1
E = mat.E
G = mat.μ
Ix = d^4 * π / 64
Iy = d^4 * π / 64
Ip = d^4 * π / 32
κ = 4 / 5
A = d^2 * π / 4;

In [377]:
Du = [κ*G*A 0 0
      0 κ*G*A 0
      0  0  E*A]

Dφ = [E*Ix 0 0
      0 E*Iy 0
      0  0  G*Ip]

D = cat(Du, Dφ, dims=(1,2))

6×6 Matrix{Float64}:
 48332.2      0.0  0.0          0.0      0.0      0.0
     0.0  48332.2  0.0          0.0      0.0      0.0
     0.0      0.0  1.5708e5     0.0      0.0      0.0
     0.0      0.0  0.0       9817.48     0.0      0.0
     0.0      0.0  0.0          0.0   9817.48     0.0
     0.0      0.0  0.0          0.0      0.0   7551.91

In [378]:
B = Bu + Bφ
K = ∫(B' ⋅ D ⋅ B, Γ="beam");
K[:,:]

186×186 SparseArrays.SparseMatrixCSC{Float64, Int64} with 810 stored entries:
⎡⠑⢄⠈⠃⠀⠀⠀⠐⠴⠄⠀⠀⠀⢀⣀⠀⠀⠀⠀⠀⠪⣆⠐⠁⠀⠀⠀⠀⠔⠀⠀⠀⠀⢀⡀⠀⠀⠀⠀⠀⎤
⎢⠦⠀⠱⣦⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠤⠀⢁⡠⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⎥
⎢⠀⠀⠀⠙⢻⣶⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⢖⡕⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢀⠀⠀⠀⠀⠉⢿⣷⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢎⠔⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠐⠇⠀⠀⠀⠀⠀⠈⠱⣦⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠨⠃⠀⠀⠀⠀⠀⠉⡠⢔⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢿⣷⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⡱⣁⢄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⢀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠿⣧⠄⠀⠀⠀⠀⠀⠀⢀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠐⠵⡡⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠘⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠁⢻⣶⡄⠀⠀⠀⠀⠋⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⠀⢖⡕⡀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢿⣷⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢎⠔⡀⠀⎥
⎢⠀⠀⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠿⣧⠀⠀⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢮⠔⎥
⎢⠪⢦⠀⠃⠀⠀⠀⠀⠦⠂⠀⠀⠀⢀⡤⠀⠀⠀⠀⠀⠑⢄⠈⠂⠀⠀⠀⠐⠶⠀⠀⠀⠀⢠⣄⠀⠀⠀⠀⠀⎥
⎢⠔⠀⠁⡰⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⠢⠀⢱⣶⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⎥
⎢⠀⠀⠀⠈⢜⠵⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠿⣧⣄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠈⢊⠕⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢀⠀⠀⠀⠀⠙⢻⣶⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠐⠁⠀⠀⠀⠀⠀⠈⢀⢎⢄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⠃⠀⠀⠀⠀⠀⠉⢵⣷⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⠕⢪⢂⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠘⢻⣶⡄⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⢀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠑⠕⡣⠂⠀⠀⠀⠀⠀⠀⣀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢿⣷⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠈⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⢜⠵⡄⠀⠀⠀⠀⠙⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠿⣧⣄⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢊⠕⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢻⣶⡄⠀⎥
⎣⠀⠀⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠈⢊⠗⠀⠀⡄⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠉⢿⣷⎦

In [379]:
bc1 = BoundaryCondition("A", field=U, ux=0, uy=0, uz=0)
bc2 = BoundaryCondition("A", field=Φ, φx=0, φy=0, φz=0)

BoundaryCondition("A", Problem("beam-3D", :VectorField, 3, 3, Material[Material("beam", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)], 1.0, 31, LowLevelFEM.Geometry("", "", 0, 0, nothing, nothing, nothing, nothing), :φ, :m, false, :lagrange, LowLevelFEM.ApproximationData(nothing, nothing, nothing)), Dict{Symbol, Union{Function, Number, ScalarField}}(:φz => 0, :φy => 0, :φx => 0))

In [380]:
fu = ∫(U ⋅ [0,0,0], Γ="D")
fφ = ∫(Φ ⋅ [0,0,10], Γ="D")

F = SystemVector([fu, fφ]);

In [381]:
u, φ = solveField(K, F, support=[bc1,bc2]);

In [382]:
showDoFResults(fu, name="force", visible=true)
showDoFResults(fφ, name="moment", visible=true)
showDoFResults(u, name="u", factor=2, visible=true);

## Recovery of beam resultants

For post-processing, the local in-plane normal vector is stored as a `VectorField`. The generalized beam strains are then recovered from the solved displacement and rotation fields.


In [383]:
#n = VectorField([nx, ny, 0]);

The displacement field is expanded to the 3D vector representation used by the general gradient and vector-algebra routines. The rotation field is converted to an elementwise field before its gradient is evaluated.

The recovered quantities are

$$
\varepsilon=\mathbf{t}\cdot\left((\nabla\mathbf{u})\mathbf{t}\right),
$$

$$
\gamma=\mathbf{n}\cdot\left((\nabla\mathbf{u})\mathbf{t}\right)-\varphi,
$$

and

$$
\kappa_b=\nabla\varphi\cdot\mathbf{t}.
$$

The corresponding internal forces are

$$
N=EA\varepsilon,
\qquad
T=\kappa_sGA\gamma,
\qquad
M=EI_z\kappa_b.
$$


In [384]:
#=
u = expandTo3D(u)
φ = nodesToElements(φ)

g = ∇(u)

ε = t ⋅ (g * t)
γ = n ⋅ (g * t) - φ

κb = ∇(φ) ⋅ t

N = E * A * ε
T = κ * G * A * γ
Mh = E * Iz * κb;
=#

In [385]:
#=
N0, plotN1 = plotOnBeam("beam", N, name="N1 graph", plot=true)
T0, plotT1 = plotOnBeam("beam", T, name="T1 graph", plot=true)
Mh0, plotMh1 = plotOnBeam("beam", Mh, name="Mh1 graph", plot=true)

N1, plotN2 = plotOnBeam("CDE", N, name="N2 graph", plot=true)
T1, plotT2 = plotOnBeam("CDE", T, name="T2 graph", plot=true)
Mh1, plotMh2 = plotOnBeam("CDE", Mh, name="Mh2 graph", plot=true)

N2, plotN3 = plotOnBeam("EFG", N, name="N3 graph", plot=true)
T2, plotT3 = plotOnBeam("EFG", T, name="T3 graph", plot=true)
Mh2, plotMh3 = plotOnBeam("EFG", Mh, name="Mh3 graph", plot=true);
=#

The returned diagram coordinates can also be processed with an external plotting package if a publication-style figure is preferred.


In [386]:
#plot(plotN1)

In [387]:
#plot(plotT1)

In [388]:
#plot(plotMh1)

In [389]:
openPostProcessor()